# Ropedia Academy — A9 · Paper deep-dive & reproduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A9.ipynb)

> **Runs a real SMPLify optimization with an ablation and plots reprojection-vs-iteration for fits with and without the pose prior.**
>
> 运行真实的 SMPLify 优化并做消融，画出有/无姿态先验时重投影随迭代的曲线。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and **visualizes the result with matplotlib** (the plot renders inline below the cell), so you learn the concept by executing and *seeing* it.

Colab's default runtime already includes `torch`, `numpy`, `networkx`, and `matplotlib`, so just press **Run all** — every cell goes green and a figure appears. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A9

In [ ]:
import torch, matplotlib.pyplot as plt

# SMPLify: FIT pose to one clip by optimization. Toy joints = base + theta (runs here).
base = torch.randn(15, 3)
joints  = lambda th: base + th
project = lambda J: J[:, :2]
kp2d = project(joints(torch.randn(15, 3))) + 0.01*torch.randn(15, 2)   # observed 2D

def fit(prior_w):                                # minimize reprojection + pose prior
    th = torch.zeros(15, 3, requires_grad=True); opt = torch.optim.Adam([th], lr=0.1); hist = []
    for _ in range(300):
        opt.zero_grad()
        reproj = ((project(joints(th)) - kp2d)**2).mean()
        (reproj + prior_w*(th**2).mean()).backward(); opt.step(); hist.append(reproj.item())
    return hist
h_prior, h_none = fit(1e-2), fit(0.0)            # ABLATION: with vs without the prior
print(f"final reproj  with prior={h_prior[-1]:.4f}  no prior={h_none[-1]:.4f}")

# Visualize the optimization: reprojection error vs iteration for both.
plt.figure(figsize=(6, 3))
plt.plot(h_prior, label="with pose prior"); plt.plot(h_none, label="no prior")
plt.yscale('log'); plt.xlabel("iteration"); plt.ylabel("reprojection error")
plt.title("SMPLify: the prior breaks depth ties"); plt.legend(); plt.show()

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A9
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks